# Predicciones ARIMA filtradas — ETFs con señal predictiva real

## Contexto: predicción de retornos para Markowitz

En la optimización de carteras de Markowitz, el vector de retornos esperados **μ** es el input más crítico y el más sensible a errores. Si el modelo de predicción no consigue captar el comportamiento del activo, alimentar Markowitz con esas predicciones es **contraproducente**: introduce ruido sesgado que distorsiona los pesos óptimos más de lo que lo haría usar simplemente cero.

### ¿Por qué el benchmark de ceros es el más exigente?

En finanzas los retornos semanales oscilan en torno a cero con media muy próxima a cero. Predecir siempre cero ya captura esa propiedad estadística central. Un modelo que no consigue batir este benchmark no está aportando ninguna señal direccional útil: sus predicciones son peores que ignorar el activo.

### Dos enfoques posibles

Para los ETFs cuyo ARIMA **no** bate al benchmark de ceros (RMSE_ARIMA ≥ RMSE_ZEROS) existen dos estrategias:

| Enfoque | Descripción | Pros | Contras |
|---------|-------------|------|---------|
| **Exclusión** | Eliminar estos ETFs del universo de inversión | Análisis limpio, solo activos con señal | Reduce diversificación, pierde activos que sí aportan en covarianza |
| **Shrinkage** | Sustituir la predicción por 0 para estos ETFs | Preserva el universo completo, prior neutro cuando no hay señal | Más complejo de implementar y explicar |

### Recomendación para el TFM

Se recomienda el **shrinkage**: cuando ARIMA no bate a ceros, se usa μ̂=0 para ese ETF en lugar de la predicción ruidosa. Este enfoque es el más utilizado en la práctica institucional (Black-Litterman, por ejemplo, combina views con un prior) y es académicamente más sólido porque:

1. **Preserva la diversificación**: el ETF sigue en la cartera si su contribución vía covarianza es positiva.
2. **Prior neutro coherente**: decir μ̂=0 equivale a «no tengo señal» — no a «el ETF va a caer».
3. **Robustez**: evita la inestabilidad de Markowitz ante pequeños cambios en μ.

Este notebook analiza los **ETFs que sí superan al benchmark de ceros** como referencia para ver qué activos tienen señal predictiva real. El capítulo de Markowitz implementará shrinkage usando 0 para los demás.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

DATA_DIR  = '../../Datos_csv'
OUT_PREDS = os.path.join(DATA_DIR, 'predicciones_arima_semanal.csv')
OUT_FILT  = os.path.join(DATA_DIR, 'predicciones_arima_semanal_filtrado.csv')

GRUPOS_ETF = {
    'Core US':         ['SPY','IVV','VOO','QQQ','DIA','IWM','MDY','IJR','ITOT','VTI'],
    'Factores':        ['MTUM','QUAL','USMV','VLUE','IWF','IWD','VUG','VTV','VIG','DVY','SCHD'],
    'Sectores':        ['XLK','XLF','XLV','XLY','XLP','XLE','XLI','XLB','XLU','XLRE','VNQ'],
    'Internacional':   ['EFA','IEFA','VEA','EEM','IEMG','VWO','EWJ','EWG','EWQ','EWU','EWT','EWZ','FXI','MCHI','INDA'],
    'Renta fija':      ['AGG','BND','LQD','HYG','JNK','TLT','IEF','SHY','TIP'],
    'Materias primas': ['GLD','IAU','SLV','USO','DBC'],
}
ORDEN_GRUPOS  = list(GRUPOS_ETF.keys())
etf_to_grupo  = {t: g for g, tickers in GRUPOS_ETF.items() for t in tickers}

print(f'CSV predicciones: {OUT_PREDS}')
print(f'CSV filtrado salida: {OUT_FILT}')

## Carga de predicciones y cálculo de métricas por ETF

In [ ]:
df_all = pd.read_csv(OUT_PREDS)
print(f'Filas totales: {len(df_all)} | ETFs: {df_all["Target"].nunique()}')
display(df_all.head(3))

# Métricas por ETF (ya están en el CSV como columnas de resumen, usamos la primera fila de cada ETF)
df_etf = (
    df_all.groupby('Target', sort=False)
    .first()   # la RMSE/MAE es la misma en todas las filas del mismo ETF
    .reset_index()
    .rename(columns={
        'Target':       'ETF',
        'RMSE':         'RMSE_ARIMA',
        'RMSE_baseline':'RMSE_BASE',
        'RMSE_zeros':   'RMSE_ZEROS',
        'MAE':          'MAE_ARIMA',
        'MAE_baseline': 'MAE_BASE',
        'MAE_zeros':    'MAE_ZEROS',
    })
    [['ETF','RMSE_ARIMA','RMSE_BASE','RMSE_ZEROS','MAE_ARIMA','MAE_BASE','MAE_ZEROS']]
)

df_etf['Grupo']           = df_etf['ETF'].map(etf_to_grupo).fillna('Otros')
df_etf['Mejora_vs_base']  = (df_etf['RMSE_BASE']  - df_etf['RMSE_ARIMA']) / df_etf['RMSE_BASE']  * 100
df_etf['Mejora_vs_zeros'] = (df_etf['RMSE_ZEROS'] - df_etf['RMSE_ARIMA']) / df_etf['RMSE_ZEROS'] * 100
df_etf['Bate_base']  = (df_etf['RMSE_ARIMA'] < df_etf['RMSE_BASE']).astype(int)
df_etf['Bate_zeros'] = (df_etf['RMSE_ARIMA'] < df_etf['RMSE_ZEROS']).astype(int)

n_total     = len(df_etf)
n_bate_zero = df_etf['Bate_zeros'].sum()
print(f'\nETFs totales     : {n_total}')
print(f'Baten zeros      : {n_bate_zero}/{n_total}')
print(f'No baten zeros   : {n_total - n_bate_zero}/{n_total}')

## ETFs que superan al benchmark de ceros

Se filtran los ETFs cuyo RMSE_ARIMA es estrictamente inferior al RMSE_ZEROS. Estos son los activos para los que el modelo ARIMA aporta señal predictiva real y cuyas predicciones pueden usarse directamente en Markowitz.

In [ ]:
df_filtrado = df_etf[df_etf['Bate_zeros'] == 1].copy()
etfs_filtrados = df_filtrado['ETF'].tolist()

print(f'ETFs con señal predictiva (baten zeros): {len(etfs_filtrados)}')
print('\nLista ETFs seleccionados:')
print(', '.join(sorted(etfs_filtrados)))

# Tabla resumen comparativa: todos vs filtrados
comp = pd.DataFrame({
    'Universo completo (58 ETFs)': [
        df_etf['RMSE_ARIMA'].mean(),
        df_etf['RMSE_ZEROS'].mean(),
        df_etf['Mejora_vs_zeros'].mean(),
        df_etf['Bate_zeros'].sum()
    ],
    f'ETFs filtrados ({len(etfs_filtrados)} ETFs)': [
        df_filtrado['RMSE_ARIMA'].mean(),
        df_filtrado['RMSE_ZEROS'].mean(),
        df_filtrado['Mejora_vs_zeros'].mean(),
        len(df_filtrado)
    ],
}, index=['RMSE_ARIMA medio', 'RMSE_ZEROS medio', 'Mejora vs zeros (%)', 'N ETFs'])

print('\nComparativa antes/después del filtro:')
display(comp.style.format({
    'Universo completo (58 ETFs)':        lambda x: f'{x:.5f}' if isinstance(x, float) and x < 100 else f'{x:.2f}' if isinstance(x, float) else str(int(x)),
    f'ETFs filtrados ({len(etfs_filtrados)} ETFs)': lambda x: f'{x:.5f}' if isinstance(x, float) and x < 100 else f'{x:.2f}' if isinstance(x, float) else str(int(x)),
}))

## ETFs excluidos (no baten zeros)

Estos son los activos para los que se aplicará **shrinkage** (μ̂=0) en el modelo de Markowitz.

In [ ]:
df_excluidos = df_etf[df_etf['Bate_zeros'] == 0].copy()
df_excluidos['Grupo'] = df_excluidos['ETF'].map(etf_to_grupo).fillna('Otros')

print(f'ETFs con shrinkage (no baten zeros): {len(df_excluidos)}')

cols_show = ['ETF','Grupo','RMSE_ARIMA','RMSE_ZEROS','Mejora_vs_zeros']
display(
    df_excluidos[cols_show]
    .sort_values('Mejora_vs_zeros')
    .reset_index(drop=True)
    .style
    .format({'RMSE_ARIMA': '{:.5f}', 'RMSE_ZEROS': '{:.5f}', 'Mejora_vs_zeros': '{:.2f} %'})
    .highlight_max(subset=['Mejora_vs_zeros'], color='#ffcccc')
)

## Análisis de resultados — ETFs filtrados

### Resumen global

In [ ]:
tabla_global = pd.DataFrame({
    'RMSE': [df_filtrado['RMSE_ARIMA'].mean(), df_filtrado['RMSE_BASE'].mean(),  df_filtrado['RMSE_ZEROS'].mean()],
    'MAE':  [df_filtrado['MAE_ARIMA'].mean(),  df_filtrado['MAE_BASE'].mean(),   df_filtrado['MAE_ZEROS'].mean()],
}, index=['ARIMA', 'BASE', 'ZEROS'])

print('Media global — ETFs filtrados (baten zeros)')
display(tabla_global.style.format('{:.6f}').highlight_min(color='#c6efce', axis=0))

print(f'\nMejora media vs BASE  : {df_filtrado["Mejora_vs_base"].mean():.2f}%')
print(f'Mejora media vs ZEROS : {df_filtrado["Mejora_vs_zeros"].mean():.2f}%')
print(f'Baten BASE  : {df_filtrado["Bate_base"].sum()}/{len(df_filtrado)}')
print(f'Baten ZEROS : {df_filtrado["Bate_zeros"].sum()}/{len(df_filtrado)} (por definición = todos)')

### Análisis por grupo de activos

In [ ]:
ORDEN_GRUPOS_VALIDOS = [g for g in ORDEN_GRUPOS if g in df_filtrado['Grupo'].unique()]

tabla_grupos = (
    df_filtrado.groupby('Grupo')
    .agg(
        N             =('ETF',             'count'),
        RMSE_ARIMA    =('RMSE_ARIMA',      'mean'),
        RMSE_BASE     =('RMSE_BASE',       'mean'),
        RMSE_ZEROS    =('RMSE_ZEROS',      'mean'),
        MAE_ARIMA     =('MAE_ARIMA',       'mean'),
        Mejora_base   =('Mejora_vs_base',  'mean'),
        Mejora_zeros  =('Mejora_vs_zeros', 'mean'),
        Bate_base     =('Bate_base',       'sum'),
        Bate_zeros    =('Bate_zeros',      'sum'),
    )
    .reindex(ORDEN_GRUPOS_VALIDOS)
    .reset_index()
)
tabla_grupos['Bate_base_str']  = tabla_grupos.apply(lambda r: f'{int(r.Bate_base)}/{int(r.N)}',  axis=1)
tabla_grupos['Bate_zeros_str'] = tabla_grupos.apply(lambda r: f'{int(r.Bate_zeros)}/{int(r.N)}', axis=1)

fmt = {c: '{:.5f}' for c in ['RMSE_ARIMA','RMSE_BASE','RMSE_ZEROS','MAE_ARIMA']}
fmt.update({'Mejora_base': '{:.2f} %', 'Mejora_zeros': '{:.2f} %'})

print('POR GRUPO — ETFs filtrados')
display(
    tabla_grupos[['Grupo','N','RMSE_ARIMA','RMSE_BASE','RMSE_ZEROS',
                  'MAE_ARIMA','Mejora_base','Mejora_zeros','Bate_base_str','Bate_zeros_str']]
    .rename(columns={
        'Bate_base_str':  'Bate base',
        'Bate_zeros_str': 'Bate zeros',
        'Mejora_base':    'Mejora vs base (%)',
        'Mejora_zeros':   'Mejora vs zeros (%)',
    })
    .style.format(fmt)
    .highlight_min(subset=['RMSE_ARIMA'], color='#c6efce')
    .highlight_max(subset=['RMSE_ARIMA'], color='#ffc7ce')
)

### Tablas por ETF dentro de cada grupo

In [ ]:
fmt_etf = {
    'RMSE_ARIMA':      '{:.5f}',
    'RMSE_BASE':       '{:.5f}',
    'RMSE_ZEROS':      '{:.5f}',
    'MAE_ARIMA':       '{:.5f}',
    'Mejora_vs_base':  '{:.2f} %',
    'Mejora_vs_zeros': '{:.2f} %',
}
cols_etf = ['ETF','RMSE_ARIMA','RMSE_BASE','RMSE_ZEROS',
            'MAE_ARIMA','Mejora_vs_base','Mejora_vs_zeros','Bate_base']

for grupo in ORDEN_GRUPOS:
    sub = df_filtrado[df_filtrado['Grupo'] == grupo][cols_etf].sort_values('RMSE_ARIMA').reset_index(drop=True)
    if sub.empty:
        continue
    print(f'\n{grupo} ({len(sub)} ETFs con señal)')
    display(
        sub.style
        .format(fmt_etf)
        .highlight_min(subset=['RMSE_ARIMA'], color='#c6efce')
        .highlight_max(subset=['RMSE_ARIMA'], color='#ffc7ce')
    )

### Visualización — ranking de mejora vs zeros (ETFs filtrados)

In [ ]:
df_sorted = df_filtrado.sort_values('Mejora_vs_zeros', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, max(8, len(df_filtrado) * 0.4)))

for ax, col, titulo in zip(
    axes,
    ['Mejora_vs_base', 'Mejora_vs_zeros'],
    ['ARIMA vs BASE (naive t-1)', 'ARIMA vs ZEROS']
):
    colores = ['#4CAF50' if v >= 0 else '#F44336' for v in df_sorted[col]]
    ax.barh(df_sorted['ETF'], df_sorted[col], color=colores)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Mejora RMSE (%)')
    ax.set_title(titulo, fontsize=11)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle(
    f'Ranking ETFs con señal predictiva — ARIMA semanal ({len(df_filtrado)} ETFs)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

## Guardar predicciones filtradas

Se guarda un CSV con las predicciones semanales únicamente de los ETFs que superan al benchmark de ceros. Este es el dataset que se usará directamente en Markowitz para los ETFs con señal. Los ETFs excluidos recibirán μ̂=0 (shrinkage).

In [ ]:
df_preds_filt = df_all[df_all['Target'].isin(etfs_filtrados)].copy()

df_preds_filt.to_csv(OUT_FILT, index=False, encoding='utf-8-sig')

print(f'Guardado: {OUT_FILT}')
print(f'Filas    : {len(df_preds_filt)}')
print(f'ETFs     : {df_preds_filt["Target"].nunique()}')
display(df_preds_filt.head(6))